# JupyterLite 教育ベンチマーク（総合版）

大学初年次 Python 入門講義を想定した、**全ベンチマーク統合ノートブック**です。
ノートブック内から測定できるものをすべて 1 つにまとめています。

| 柱 | 内容 | このノートで測れるか |
|---|---|---|
| **A 初回体験** | ページロード時間・DL量・端末情報 | △ ブラウザ API 経由で**一部**取得（真の cold load は外部ハーネス） |
| **B 信頼性** | ストレージ容量・FS 入出力 | △ 容量と FS は取得（タブ閉じ/別端末の消失は手動試験） |
| **C 網羅/実行時間** | import・計算・描画 | ○ 全部取得 |

- すべて **Pyodide（ネイティブ依存なし）**で完走するよう設計。JS API が無い環境でも `try/except` で素通りします。
- 最初のコードセルは `hello` を出力（自動計測ハーネスの検出ターゲット）。
- 最後に全結果を JSON として表示し `bench_results.json` に保存します。

**使い方**: Run → Run All Cells（または計測ハーネスが Run All を発火）。

In [ ]:
print("hello")

# このセルが走った時点での「ナビゲーション開始からの経過 ms」を記録する。
# Run All の起点付近で取得することで、cold load + カーネル初期化の概算プロキシになる。
try:
    import js
    _PAGE_MS_AT_START = float(js.performance.now())
except Exception:
    _PAGE_MS_AT_START = None
_PAGE_MS_AT_START

## 0. 計測フックの準備

In [ ]:
import time

BENCH = []

class timer:
    """with timer("ラベル"): ... で囲んだ処理の実行時間を記録する。"""
    def __init__(self, label):
        self.label = label
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *exc):
        dt = time.perf_counter() - self.t0
        BENCH.append({"label": self.label, "seconds": round(dt, 4)})
        print(f"[bench] {self.label}: {dt:.4f}s")
        return False

## A. 初回ロード / 環境（ブラウザ API から取得できる範囲）

`performance` と `navigator` から、ページロード時間・概算ダウンロード量・端末情報を取得する。

> **注意**: ダウンロード量は `transferSize` の合計だが、CDN（別オリジン）から配信される
> Pyodide 資産は `Timing-Allow-Origin` ヘッダが無いと `transferSize` が 0 になり、**過小評価**され得る。
> 真の cold load（DL + 初期化の壁時計）は外部の Playwright ハーネスで測ること。

In [ ]:
import sys

ENV = {
    "python_version": sys.version.split()[0],
    "platform": sys.platform,
    "is_pyodide": ("pyodide" in sys.modules) or (sys.platform == "emscripten"),
    "user_agent": None,
    "hardware_concurrency": None,
    "device_memory_gb": None,
}
try:
    import js
    ENV["user_agent"] = str(js.navigator.userAgent)
    try:
        ENV["hardware_concurrency"] = int(js.navigator.hardwareConcurrency)
    except Exception:
        pass
    try:
        # deviceMemory は一部ブラウザ（Firefox/Safari 等）に無い
        ENV["device_memory_gb"] = float(js.navigator.deviceMemory)
    except Exception:
        pass
except Exception as e:
    print(f"[env] ブラウザ情報を取得できません: {e!r}")

ENV

In [ ]:
PAGE_LOAD = {
    "ms_since_navigation_at_first_cell": _PAGE_MS_AT_START,
    "dom_content_loaded_ms": None,
    "load_event_end_ms": None,
    "response_end_ms": None,
    "estimated_download_mb": None,
    "resource_count": None,
    "caveat": "CDN(別オリジン)資産は Timing-Allow-Origin が無いと transferSize=0 となり DL量を過小評価し得る",
}
try:
    import js
    navs = js.performance.getEntriesByType("navigation")
    if navs.length > 0:
        nav = navs[0]
        PAGE_LOAD["dom_content_loaded_ms"] = round(float(nav.domContentLoadedEventEnd), 1)
        PAGE_LOAD["load_event_end_ms"] = round(float(nav.loadEventEnd), 1)
        PAGE_LOAD["response_end_ms"] = round(float(nav.responseEnd), 1)

    total = 0.0
    n = 0
    for e in js.performance.getEntriesByType("resource"):
        try:
            total += float(e.transferSize)
        except Exception:
            pass
        n += 1
    PAGE_LOAD["estimated_download_mb"] = round(total / 1e6, 2)
    PAGE_LOAD["resource_count"] = n
except Exception as e:
    print(f"[page] ロードタイミングを取得できません: {e!r}")

PAGE_LOAD

## B. ストレージ / 永続性プローブ

ストレージ容量と仮想ファイルシステムの入出力を測る。

> **注意**: 「タブを閉じる / ブラウザ再起動 / 別端末 / キャッシュ消去 / プライベートモード」での
> データ消失は**ノートブック内からは検証できない**。これらは手動シナリオ試験で確認すること
> （計画書の柱 B チェックリスト参照）。ここで分かるのは容量と FS が動くかどうかまで。

In [ ]:
import time, os

STORAGE = {
    "quota_mb": None,
    "usage_mb": None,
    "persisted": None,
    "fs_roundtrip_s": None,
    "note": "タブ閉じ/別端末/キャッシュ消去による消失はノート内では検証不可（手動試験で確認）",
}

# ブラウザのストレージ見積り（非同期）
try:
    import js
    est = await js.navigator.storage.estimate()
    STORAGE["quota_mb"] = round(float(est.quota) / 1e6, 1)
    STORAGE["usage_mb"] = round(float(est.usage) / 1e6, 3)
except Exception as e:
    print(f"[storage] estimate を取得できません: {e!r}")

try:
    import js
    STORAGE["persisted"] = bool(await js.navigator.storage.persisted())
except Exception:
    pass

# 仮想ファイルシステムの書き込み→読み出し往復
try:
    blob = "x" * 100_000
    t0 = time.perf_counter()
    with open("fs_test.txt", "w") as f:
        f.write(blob)
    with open("fs_test.txt") as f:
        back = f.read()
    STORAGE["fs_roundtrip_s"] = round(time.perf_counter() - t0, 5)
    assert back == blob
    os.remove("fs_test.txt")
except Exception as e:
    print(f"[storage] FS 往復に失敗: {e!r}")

STORAGE

## C-1. Python 基礎（変数・制御構文・関数）

純 Python。入門講義前半の典型。どの環境でも一瞬で動くはず。

In [ ]:
with timer("python basics"):
    total = 0
    for i in range(1, 101):
        if i % 2 == 0:
            total += i

    def greet(name):
        return f"Hello, {name}!"

    result = {"evens_sum_1_to_100": total, "greeting": greet("students")}

result

## C-2. NumPy（配列と基本統計）

In [ ]:
with timer("import numpy"):
    import numpy as np

with timer("numpy ops"):
    a = np.arange(1_000_000, dtype=np.float64)
    stats = {"mean": float(a.mean()), "sum": float(a.sum()), "std": float(a.std())}

stats

## C-3. pandas（小さな CSV の読み込みと集計）

CSV はノート内で書き出してから読み込み、同梱ファイルに依存せず自己完結させる。

In [ ]:
csv_text = """name,group,score
Alice,A,82
Bob,B,75
Carol,A,90
Dave,B,68
Eve,A,88
Frank,B,79
"""

with open("sample.csv", "w") as f:
    f.write(csv_text)

with timer("import pandas"):
    import pandas as pd

with timer("read_csv + groupby"):
    df = pd.read_csv("sample.csv")
    summary = df.groupby("group")["score"].mean()

summary

## C-4. SymPy（記号計算・任意）

入門講義で記号計算に触れる場合の指標。未対応環境ならスキップして完走する。

In [ ]:
try:
    with timer("import sympy"):
        import sympy as sp

    with timer("sympy factorint"):
        factors = sp.factorint(2**67 - 1)

    with timer("sympy expand"):
        x = sp.symbols("x")
        _ = sp.expand((x + 1) ** 50)

    print("[sympy] OK  factor(2^67 - 1) =", factors)
except Exception as e:
    print(f"[sympy] スキップ/失敗: {e!r}")

## C-5. matplotlib（可視化）

In [ ]:
with timer("import matplotlib"):
    import matplotlib.pyplot as plt

import numpy as np

with timer("line plot"):
    fig, ax = plt.subplots()
    ax.plot(range(10), [x**2 for x in range(10)], marker="o")
    ax.set_title("y = x^2")
    plt.show()

with timer("histogram"):
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots()
    ax.hist(rng.normal(size=1000), bins=30)
    ax.set_title("normal sample (n=1000)")
    plt.show()

## C-6. ipywidgets（対話要素・任意）

このセルが `ModuleNotFoundError` になる場合、デプロイに ipywidgets が含まれていない。
パッケージ追加だけでなく、JupyterLite 側にウィジェットのフロントエンド拡張も含める必要がある点に注意。

In [ ]:
try:
    with timer("ipywidgets interact"):
        from ipywidgets import interact
        import numpy as np
        import matplotlib.pyplot as plt

        def plot_sine(freq=1.0):
            x = np.linspace(0, 2 * np.pi, 200)
            plt.figure()
            plt.plot(x, np.sin(freq * x))
            plt.title(f"sin({freq} x)")
            plt.show()

        interact(plot_sine, freq=(0.5, 5.0, 0.5))
    print("[ipywidgets] OK")
except Exception as e:
    print(f"[ipywidgets] スキップ/失敗: {e!r}")

## 結果の集計・出力

柱 A / B / C をまとめて JSON で表示し、`bench_results.json` に保存する
（JupyterLite では左のファイルブラウザからダウンロード可能）。端末・ブラウザごとに回収する。

In [ ]:
import json, time

payload = {
    "schema": "jupyterlite-edu-bench/v2",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "environment": ENV,        # 柱A: 端末・ブラウザ
    "page_load": PAGE_LOAD,    # 柱A: ロード時間・DL量
    "storage": STORAGE,        # 柱B: 容量・FS
    "measurements": BENCH,     # 柱C: import/計算/描画
}

text = json.dumps(payload, ensure_ascii=False, indent=2)
print(text)

try:
    with open("bench_results.json", "w") as f:
        f.write(text)
    print()
    print("[saved] bench_results.json （ファイルブラウザからダウンロード可）")
except Exception as e:
    print()
    print(f"[warn] 保存失敗: {e!r}")